# justounetsimple_opt on the FPGA

The optimized kernel: the same quantized 2D-JustoUNet as `justounetsimple`, but the
whole capture in one start, all of the work in hardware. The kernel reads the raw L1a
cube straight from DDR, drops the 10 unused bands, z-scores the rest, cuts the image
into 32 x 32 patches (its last row and column repeated to fill the edge patches) and
writes every pixel's 3 scores and its class into place. The ARM only starts it. Inside,
a DATAFLOW chain: four DDR readers fill one on-chip patch buffer while the other feeds
the four conv blocks (ping-pong), all in integer fixed point. The answers are checked
bit for bit against the PC's integer model of the kernel; against the float model,
the int8 input moves a few pixels' scores, but hardly any class.

Cloud / land / sea classification of a HYPSO-2 capture: highest score = class:
**0 cloud, 1 land, 2 sea**.

Next to this notebook: `justounetsimple_opt.bit` + `.hwh`, `xjustounetsimple_opt_hw.h`,
and an image zip made on the PC with
`python3 tools/justounetsimple_image.py prepare ... --out aegean_unet` and `fpga_zip aegean_unet`
(the same image works for both kernels). Run the cells from the top.

## 1. Load the bitstream

In [ ]:
import json, re, time, zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

overlay = Overlay('justounetsimple_opt.bit')   # also sets the FPGA clock from the .hwh
kernel = overlay.kernel                         # the HLS block, named 'kernel' in the block design

header = Path('xjustounetsimple_opt_hw.h').read_text()
def register(name):
    return int(re.search(rf'_ADDR_{name}\s+(0x[0-9a-fA-F]+)', header).group(1), 16)
AP_CTRL, HEIGHT, WIDTH = register('AP_CTRL'), register('HEIGHT_DATA'), register('WIDTH_DATA')
DINS = [register(f'DIN{i}_DATA') for i in range(4)]
DOUT = register('DOUT_DATA')

# The kernel's formats (justounetsimple_opt.hpp): in, the raw L1a cube as captured
# (H x W x 120 uint16); out, H x W x 4 int32: the 3 scores = score * 2^OUT_FRAC, then
# the class (0 cloud, 1 land, 2 sea).
BANDS, OUT_FRAC, LANES = 120, 16, 4

def launch(cube_address, scores_address, height, width):
    # Start the kernel on a height x width cube; all four input ports read it.
    for offset, address in [(d, cube_address) for d in DINS] + [(DOUT, scores_address)]:
        kernel.write(offset, address & 0xFFFFFFFF)
        kernel.write(offset + 4, address >> 32)
    kernel.write(HEIGHT, height)
    kernel.write(WIDTH, width)
    kernel.write(AP_CTRL, 1)

def wait(timeout=60):
    deadline = time.monotonic() + timeout
    while not kernel.read(AP_CTRL) & 0x2:      # done
        if time.monotonic() > deadline:
            raise TimeoutError('kernel did not finish')

def to_float(scores):
    return np.multiply(np.asarray(scores), np.float32(2.0 ** -OUT_FRAC), dtype=np.float32)

print('kernel idle:', bool(kernel.read(AP_CTRL) & 0x4))

## 2. Load the image into DDR

The zip holds two answers from the PC: `reference_fixed.npy`, what the kernel must
write, bit for bit (the integer model), and `reference_scores.npy`, the float model's
scores. An image zip from before the kernel's input became int8 lacks the first; make
it again with `tools/justounetsimple_image.py prepare`.

The image zip is on the SD card, next to this notebook; it is unpacked there (only when
the zip is newer than what is unpacked). The raw cube then goes from the SD card into
DDR that the FPGA can read (PYNQ's CMA: 512 MB, shared by every running notebook --
shut down the ones you are not using). If the whole cube does not fit, it stays in
ordinary memory and goes to the FPGA in strips of 128 rows, two strip buffers taking
turns. The SD card is slow, so this cell takes a while; the classification does not
touch it.

In [ ]:
IMAGE = 'aegean_unet.zip'      # or an already unpacked folder
folder = Path(IMAGE)
if folder.suffix == '.zip':
    unpacked = Path(folder.stem) / 'cube.npy'
    if not unpacked.exists() or unpacked.stat().st_mtime < folder.stat().st_mtime:
        with zipfile.ZipFile(folder) as z:
            z.extractall('.')
    folder = Path(folder.stem)
t0 = time.monotonic()
cube = np.load(folder / 'cube.npy', mmap_mode='r')     # read from the SD card as it is used
if cube.dtype != np.uint16:                           # an image folder from before: counts as floats
    cube = np.rint(cube).astype(np.uint16)
reference = np.load(folder / 'reference_scores.npy')  # the numpy model's scores, from the PC
if not (folder / 'reference_fixed.npy').exists():
    raise FileNotFoundError(f'{folder}/reference_fixed.npy is missing: make the image zip again with '
                            'tools/justounetsimple_image.py prepare')
reference_fixed = np.load(folder / 'reference_fixed.npy')   # the kernel's exact output, from the PC
meta = json.loads((folder / 'meta.json').read_text())
H, W, _ = cube.shape
PATCHES = -(-H // 32) * -(-W // 32)

scores_out = allocate(shape=(H, W, LANES), dtype=np.int32)
STRIP = 128                                           # rows per strip, if the cube does not fit
try:
    cube_in = allocate(shape=cube.shape, dtype=np.uint16)
    cube_in[:] = cube
    cube_in.flush()                                   # into DDR, not just the CPU cache
    strips = None
except Exception as error:                            # not enough (contiguous) CMA
    cube_in = None
    cube = np.ascontiguousarray(cube)                 # all of it in memory, for the strips
    strips = [allocate(shape=(STRIP, W, BANDS), dtype=np.uint16) for _ in range(2)]
    print(f'the whole cube does not fit in CMA ({error}): strips of {STRIP} rows instead')
print(f'{H} x {W} pixels, {PATCHES} patches, loaded in {time.monotonic() - t0:.1f} s')

## 3. Self-test: the first patch against the PC's answer

In [ ]:
# The top-left patch on its own, in its own buffers.
patch_in = allocate(shape=(min(H, 32), min(W, 32), BANDS), dtype=np.uint16)
patch_out = allocate(shape=patch_in.shape[:2] + (LANES,), dtype=np.int32)
patch_in[:] = cube[:32, :32]
patch_in.flush()
launch(patch_in.physical_address, patch_out.physical_address, *patch_in.shape[:2])
wait()
patch_out.invalidate()
wrong = int((np.asarray(patch_out) != reference_fixed[:32, :32]).sum())
print('PASS' if wrong == 0 else 'FAIL', f'{wrong} values differ from the PC integer model')

## 4. The whole image

**Wall clock**: from calling `classify` until the ARM can read the scores and classes
(int32: score * 2^OUT_FRAC, and the class the kernel picked). **FPGA**: from the start bit
until the done bit. Turning the scores into floats, for the checks below, is timed
separately.

In [ ]:
def classify(timeout=60):
    # The loaded cube -> (H x W x 4 int32: scores and class, seconds the FPGA ran).
    if strips is None:
        t = time.perf_counter()
        launch(cube_in.physical_address, scores_out.physical_address, H, W)
        wait(timeout)
        fpga = time.perf_counter() - t
    else:
        # Double-buffered strips: the ARM copies strip s + 1 while the FPGA does strip s.
        def fill(s):
            rows = cube[s * STRIP:(s + 1) * STRIP]
            strips[s % 2][:len(rows)] = rows
            strips[s % 2].flush()
            return len(rows)
        count, fpga = -(-H // STRIP), 0.0
        rows = fill(0)
        for s in range(count):
            t = time.perf_counter()
            launch(strips[s % 2].physical_address,
                   scores_out.physical_address + s * STRIP * W * LANES * 4, rows, W)
            next_rows = fill(s + 1) if s + 1 < count else 0
            wait(timeout)
            fpga += time.perf_counter() - t           # with the copy, if that took longer
            rows = next_rows
    scores_out.invalidate()                           # read what the kernel wrote
    return np.asarray(scores_out), fpga

t = time.perf_counter()
raw_scores, fpga = classify()
wall = time.perf_counter() - t
classes = raw_scores[..., 3]
t = time.perf_counter()
scores = to_float(raw_scores[..., :3])
convert = time.perf_counter() - t
print(f'wall clock: {1000 * wall:7.1f} ms for {H} x {W} pixels, {PATCHES} patches')
print(f'FPGA:       {1000 * fpga:7.1f} ms, {1e6 * fpga / PATCHES:.1f} us per patch'
      + ('' if strips is None else f' (in strips of {STRIP} rows, with any copy that outlasted the FPGA)'))
print(f'then to floats on the ARM: {1000 * convert:.1f} ms')
np.save(folder / 'fpga_scores.npy', scores)

## 5. Check it: against the PC's answer and against the labels

In [ ]:
close = raw_scores == reference_fixed
print(f'bit-exact against the PC integer model: {close.all(-1).mean():.2%} of pixels')
print(f'same class as the PC float model: {(classes == reference.argmax(-1)).mean():.3%}, '
      f'scores within 0.1 of it: {(np.abs(scores - reference) <= 0.1).all(-1).mean():.2%}')
labels = np.load(folder / 'labels.npy') if (folder / 'labels.npy').exists() else None
if labels is not None:
    known = labels < 3
    print(f'accuracy against the labels: {(classes[known] == labels[known]).mean():.2%}')
    for c, name in enumerate(('cloud', 'land', 'sea')):
        union = ((classes == c) | (labels == c))[known].sum()
        iou = ((classes == c) & (labels == c))[known].sum() / union if union else float('nan')
        print(f'  IoU {name}: {iou:.3f}')

In [ ]:
COLORS = np.array([[235, 235, 235], [60, 140, 60], [30, 90, 180], [128, 128, 128]], dtype=np.uint8)
rgb = cube[:, :, meta['rgb_bands']].astype(np.float64)
lo, hi = np.percentile(rgb, [2, 98])
panels = {'picture': np.clip((rgb - lo) / (hi - lo), 0, 1),
          'FPGA classes': COLORS[classes]}
if labels is not None:
    panels['labels'] = COLORS[np.minimum(labels, 3)]
    wrong = np.full(labels.shape + (3,), 255, dtype=np.uint8)   # white: same class as the label
    wrong[~known] = 128                                         # gray: unlabeled
    wrong[known & (classes != labels)] = (220, 30, 30)          # red: a different class
    panels['differs from labels'] = wrong
else:
    panels['differs from PC'] = ~close.all(-1)
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4))
for ax, (title, img) in zip(axes, panels.items()):
    ax.imshow(img, cmap='gray_r' if img.ndim == 2 else None)
    ax.set_title(title)
    ax.axis('off')
plt.show()     # classes: cloud white, land green, sea blue, gray = unlabeled

## 6. Speed: the FPGA alone

The kernel again on the cube already in DDR (in strips: on one strip), best of 5.

In [ ]:
buffer, rows = (cube_in, H) if strips is None else (strips[0], min(STRIP, H))
if strips is not None:
    strips[0][:rows] = cube[:rows]
    strips[0].flush()
runs = []
for _ in range(5):
    t = time.perf_counter()
    launch(buffer.physical_address, scores_out.physical_address, rows, W)
    wait()
    runs.append(time.perf_counter() - t)
n = -(-rows // 32) * -(-W // 32)
best = min(runs)
print(f'FPGA alone: {1000 * best:.1f} ms for {rows} x {W} pixels ({n} patches), {1e6 * best / n:.1f} us per patch')
print(f'the whole image: about {1000 * best * PATCHES / n:.1f} ms on the FPGA, {1000 * wall:.1f} ms wall clock')